# Exploring the overflow issue when dealing with float('-inf') in coreML

## Can I replicate any Overflow Error during model casting process ? 

In [63]:
import numpy as np
import torch
import torch.nn as nn
import coremltools as ct
import io
from contextlib import redirect_stderr, redirect_stdout
import warnings
warnings.simplefilter("always")

In [51]:
class Sample_Model(nn.Module):

    def __init__(self,val:float = float('-inf')):
        '''
        Initialize Model with Seq Len and Fill Val
        '''
        super().__init__()
        self.val = val

    def forward(self, x: torch.Tensor):
        '''
        Simple one way forward pass
        '''
        N = x.shape[1]
        # Create tensor with -inf
        tensor = torch.full(size = (N, N), fill_value = self.val)

        '''if flag: # Use triu if flagged
            tensor = torch.triu(input = tensor, diagonal = 1)'''

        return x + tensor


In [52]:
class Triu_Model(Sample_Model):
    '''
    Child of Sample Model
    '''
    def __init__(self,val: float = float('-inf')):
        super().__init__(val)

    def forward(self, x: torch.Tensor):
        '''
        Forward pass with triu
        '''
        N = x.shape[1]
        # Create tensor with -inf
        tensor = torch.full(size = (N, N), fill_value = self.val)
        # Zero out upper triangle
        tensor = torch.triu(input = tensor, diagonal = 1)

        return x + tensor
        

In [54]:
class Softmax_Model(Sample_Model):

    def __init__(self, val: float = float('-inf')):
            super().__init__(val)

    def forward(self, x: torch.Tensor):
        '''
        Simple one way forward pass
        '''
        N = x.shape[1]
        # Create tensor with -inf
        tensor = torch.full(size = (N, N), fill_value = self.val)

        '''if flag: # Use triu if flagged
            tensor = torch.triu(input = tensor, diagonal = 1)'''

        return torch.softmax(x + tensor, dim = -1)

In [ ]:
class Softmax_Triu_Model(Sample_Model):
    '''
    Child of Sample Model
    '''
    def __init__(self, val: float = float('-inf')):
        super().__init__(val)

    def forward(self, x: torch.Tensor):
        '''
        Forward pass with triu
        '''
        N = x.shape[1]
        # Create tensor with -inf
        tensor = torch.full(size = (N, N), fill_value = self.val)
        # Zero out upper triangle
        tensor = torch.triu(input = tensor, diagonal = 1)

        return torch.softmax(x + tensor, dim = -1)

In [64]:
def export(model: Sample_Model, N: int):
    '''
    Dummy export function for root cause analysis
    '''

    # Model to eval mode
    model.eval()
    # Create Dummy Tensor for Trace
    dummy = torch.ones(1, N, dtype = torch.int32)

    # Trace Model -> Get Graph
    with torch.no_grad():
        traced = torch.jit.trace(model, dummy)
        output_tensor = model(dummy)

    # Initialize random input tensor
    seq_dim     = ct.RangeDim(lower_bound=1, upper_bound=256, default=256)
    input_shape = ct.Shape(shape=(1, seq_dim))

    mlmodel = ct.convert(
        traced,
        inputs = [ct.TensorType(name = "token_ids", shape = input_shape, dtype = np.int32)],
        outputs = [ct.TensorType(name = "logits", dtype = np.float32)],
        minimum_deployment_target = ct.target.macOS13,
        compute_units = ct.ComputeUnit.ALL
    )

    # Run Inference
    x = np.zeros((1, N), dtype = np.int32)
    out = mlmodel.predict({'token_ids':x})
    logits = out['logits']
    print(logits)
    if np.any(np.isnan(logits)):
        print(f"NaN value encountered in Logits.")
    return logits



Input Dtype is Int32; 
Output is expected to be float32 due to fill value of matrix being float

In [65]:
for _ in range(1):
    # 1. Buffer for printed output
    buffer = io.StringIO()

    # 2. Redirect Std Out when running export
    with redirect_stderr(buffer), redirect_stdout(buffer):
        logits = export(model = Sample_Model(), N = 128)

    # 3. Get Output
    output_str = buffer.getvalue()

    # 3. Check 'overflow'
    if 'overflow' in output_str:
        print(f"{output_str}")

/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1006: DeprecationWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/coremltools/converters/mil/mil/ops/defs/iOS15/elementwise_unary.py:889: RuntimeWarning: overflow encountered in cast
  return input_var.val.astype(dtype=string_to_nptype(dtype_val))
/Us

Overflow error occurs even before triu operation. The issue seems to be with input val being -inf

In [74]:
for _ in range(1):
    # 1. Buffer for printed output
    buffer = io.StringIO()

    # 2. Redirect Std Out when running export
    with redirect_stderr(buffer), redirect_stdout(buffer):
        logits = export(model = Sample_Model(val = -1e4), N = 128)

    # 3. Get Output
    output_str = buffer.getvalue()

    print(logits)

    # 3. Check 'overflow'
    if 'RuntimeWarning: overflow' in output_str:
        print(f"{output_str}")

[[-10000. -10000. -10000. ... -10000. -10000. -10000.]
 [-10000. -10000. -10000. ... -10000. -10000. -10000.]
 [-10000. -10000. -10000. ... -10000. -10000. -10000.]
 ...
 [-10000. -10000. -10000. ... -10000. -10000. -10000.]
 [-10000. -10000. -10000. ... -10000. -10000. -10000.]
 [-10000. -10000. -10000. ... -10000. -10000. -10000.]]
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1006: DeprecationWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings

In [80]:
for _ in range(1):
    # 1. Buffer for printed output
    buffer = io.StringIO()

    # 2. Redirect Std Out when running export
    with redirect_stderr(buffer), redirect_stdout(buffer):
        logits = export(model = Sample_Model(val = -1e14), N = 128)

    # 3. Get Output
    output_str = buffer.getvalue()

    print(logits)

    # 3. Check 'overflow'
    if 'RuntimeWarning: overflow' in output_str:
        print(f"{output_str}")

[[-inf -inf -inf ... -inf -inf -inf]
 [-inf -inf -inf ... -inf -inf -inf]
 [-inf -inf -inf ... -inf -inf -inf]
 ...
 [-inf -inf -inf ... -inf -inf -inf]
 [-inf -inf -inf ... -inf -inf -inf]
 [-inf -inf -inf ... -inf -inf -inf]]
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1006: DeprecationWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/Users/ashwathsreeram/minico

**Testing with Triu**

In [75]:
for _ in range(1):
    # 1. Buffer for printed output
    buffer = io.StringIO()

    # 2. Redirect Std Out when running export
    with redirect_stderr(buffer), redirect_stdout(buffer):
        logits = export(model = Triu_Model(), N = 128)

    # 3. Get Output
    output_str = buffer.getvalue()

    print(logits)

    # 3. Check 'overflow'
    if 'RuntimeWarning: overflow' in output_str:
        print(f"{output_str}")

[[ nan -inf -inf ... -inf -inf -inf]
 [ nan  nan -inf ... -inf -inf -inf]
 [ nan  nan  nan ... -inf -inf -inf]
 ...
 [ nan  nan  nan ...  nan -inf -inf]
 [ nan  nan  nan ...  nan  nan -inf]
 [ nan  nan  nan ...  nan  nan  nan]]
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1006: DeprecationWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/Users/ashwathsreeram/minico

In [76]:
for _ in range(1):
    # 1. Buffer for printed output
    buffer = io.StringIO()

    # 2. Redirect Std Out when running export
    with redirect_stderr(buffer), redirect_stdout(buffer):
        logits = export(model = Softmax_Model(), N = 128)

    # 3. Get Output
    output_str = buffer.getvalue()

    # 3. Check 'overflow'
    if 'overflow' in output_str:
        print(f"{output_str}")

/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1006: DeprecationWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/coremltools/converters/mil/mil/ops/defs/iOS15/elementwise_unary.py:889: RuntimeWarning: overflow encountered in cast
  return input_var.val.astype(dtype=string_to_nptype(dtype_val))
/Us

In [77]:
for _ in range(1):
    # 1. Buffer for printed output
    buffer = io.StringIO()

    # 2. Redirect Std Out when running export
    with redirect_stderr(buffer), redirect_stdout(buffer):
        inf_logits = export(model = Softmax_Triu_Model(), N = 128)

    # 3. Get Output
    output_str = buffer.getvalue()

    print(inf_logits)

    # 3. Check 'overflow'
    if 'overflow' in output_str:
        print(f"{output_str}")

[[nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1006: DeprecationWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-pa

In [78]:
for _ in range(1):
    # 1. Buffer for printed output
    buffer = io.StringIO()

    # 2. Redirect Std Out when running export
    with redirect_stderr(buffer), redirect_stdout(buffer):
        non_inf_logits = export(model = Softmax_Triu_Model(val = -1e4), N = 128)

    # 3. Get Output
    output_str = buffer.getvalue()

    print(non_inf_logits)

    # 3. Check 'overflow'
    if 'overflow' in output_str:
        print(f"{output_str}")

[[1.         0.         0.         ... 0.         0.         0.        ]
 [0.5        0.5        0.         ... 0.         0.         0.        ]
 [0.33325195 0.33325195 0.33325195 ... 0.         0.         0.        ]
 ...
 [0.00793457 0.00793457 0.00793457 ... 0.00793457 0.         0.        ]
 [0.00787354 0.00787354 0.00787354 ... 0.00787354 0.00787354 0.        ]
 [0.0078125  0.0078125  0.0078125  ... 0.0078125  0.0078125  0.0078125 ]]
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1006: DeprecationWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/torch/jit/_trace.py:1145: DeprecationWar

In [79]:
print(f"Exactness: {np.all(inf_logits == non_inf_logits)}")

Exactness: False


The issue is simple: During CoreML model conversion process, a fixed mask can get created if dimensions of the vector are defined. However, if flexible dims given, then the creation happens during runtime and having out of bounds values creates NaN.

## Forward Pass with PyTorch Model + '-inf' masking + CoreML Inference

Given the largely in-consistent ability to reproduce overflow error, we are moving onto performing actual forward pass for replication of errror

In [12]:
import sys
sys.path.append("/Users/ashwathsreeram/Desktop/mlx") 

In [13]:
from export.export_coreml import export
from tokenizer import CharTokenizer
import config
import coremltools as ct
import numpy as np

In [26]:
export()

/Users/ashwathsreeram/Desktop/mlx/export/model_torch.py:93: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert x.shape[-1] == self.d_model, "Input shape doesnt match Model"
/Users/ashwathsreeram/Desktop/mlx/export/model_torch.py:31: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert k_dim == self.d_model, "Model and output Dims do not match"


Loaded weights from /Users/ashwathsreeram/Desktop/mlx/weights/gpt_shakespeare.npz
JIT trace complete.


Running MIL default pipeline:  45%|████▌     | 43/95 [00:00<00:00, 426.38 passes/s]/Users/ashwathsreeram/miniconda3/envs/mlx/lib/python3.12/site-packages/coremltools/converters/mil/mil/ops/defs/iOS15/elementwise_unary.py:889: RuntimeWarning: overflow encountered in cast
  return input_var.val.astype(dtype=string_to_nptype(dtype_val))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 949.51 passes/s]


CoreML model saved to /Users/ashwathsreeram/Desktop/mlx/artifacts/gpt_shakespeare.mlpackage
Input  : token_ids  — shape (1, T),              dtype int32
Output : logits     — shape (1, T, 65), dtype float32


### Forward Pass

In [15]:
# ── Sampling ──────────────────────────────────────────────────────────────────
def sample_top_k(logits: np.ndarray, k: int, temperature: float) -> int:
    # Keep only the top-k logit positions, mask the rest to -inf
    top_k_idx        = np.argpartition(logits, -k)[-k:]
    masked           = np.full_like(logits, -1e9)
    masked[top_k_idx] = logits[top_k_idx]
    # Scale by temperature, then softmax, then draw one sample
    masked  -= masked.max()                 # numerical stability
    probs    = np.exp(masked / temperature)
    probs   /= probs.sum()
    return int(np.random.choice(len(probs), p=probs))

To Note: We encountered the overflow error again

In [16]:
# Loading Paths, Configs and Tokenizer
MODEL_PATH     = str(config.MLPACKAGE_PATH)
CONTEXT_LEN    = config.CONTEXT_LEN
MAX_NEW_TOKENS = 300

encode = CharTokenizer().encode
decode = CharTokenizer().decode

In [17]:
# Loading Model and running inference
model = ct.models.MLModel(MODEL_PATH)
print(f"Model Loaded: {MODEL_PATH}")

Model Loaded: /Users/ashwathsreeram/Desktop/mlx/artifacts/gpt_shakespeare.mlpackage


In [18]:
prompt: str = "ROMEO:"

# Tokenize String
ids = encode(prompt)

# Forward pass for max_new_tokens iteration
for tok in range(MAX_NEW_TOKENS):

    context = ids[-CONTEXT_LEN:]
    seq_len = len(context)

    x = np.zeros((1, CONTEXT_LEN), dtype = np.int32) # Creating context length buffer
    x[0, :seq_len] = context # Filling in context length 

    out = model.predict({"token_ids":x})

    # Get Logits
    logits = out['logits'] # [1, context_len, vocab]

    # We need only last pos
    last_logits = logits[0, -1, :]

    # Next Token generation through Top K sampling
    next_id = sample_top_k(logits = last_logits, k = 40, temperature = 0.8)

    print(f"tok {tok} : {next_id}")

    # Add new token pos
    ids.append(next_id)

print(decode(ids))

    


ValueError: probabilities contain NaN

In [19]:
prompt: str = "ROMEO:"

# Tokenize String
ids = encode(prompt)

# Forward pass for max_new_tokens iteration
for tok in range(MAX_NEW_TOKENS):

    context = ids[-CONTEXT_LEN:]
    seq_len = len(context)

    x = np.ones((1, CONTEXT_LEN), dtype = np.int32) # Creating context length buffer
    x[0, :seq_len] = context # Filling in context length 

    out = model.predict({"token_ids":x})

    # Get Logits
    logits = out['logits'] # [1, context_len, vocab]

    # We need only last pos
    last_logits = logits[0, -1, :]

    # Next Token generation through Top K sampling
    next_id = sample_top_k(logits = last_logits, k = 40, temperature = 0.8)

    print(f"tok {tok} : {next_id}")

    # Add new token pos
    ids.append(next_id)

print(decode(ids))

ValueError: probabilities contain NaN

Experiment Status:

The issue with NaN seems to occur during first pass itself. Without top K sampling, this wouldn't have been encountered as greedy decoding would simply output token 0 at every step. My next step is to print and understand the weights of the saved model

## Weight visualization

In [5]:
weights = np.load(file = config.WEIGHTS_PATH)

Check 1: Are any of the weights Nan ? 

In [10]:
for key, values in weights.items():

    if np.any(np.isnan(values)):    
        print(f"Layer : {key} --> {np.min(values), np.max(values)} --> {np.any(np.isnan(values))}")

None of the weights are NaN. It more and more looks like its simply an operation using the said mask